In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency, ttest_ind

# Load  Data 
df = pd.read_csv("../data/MachineLearningRating_v3.txt", sep="|")

df.columns = df.columns.str.strip()

# Drop rows with missing key columns
df = df[df['Gender'].notnull()]
df = df[df['TotalPremium'] > 0]  # avoid division by zero

# Derived metrics
df['Loss_Ratio'] = df['TotalClaims'] / df['TotalPremium']
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df['HasClaim'] = df['TotalClaims'] > 0

# Optional: Cast PostalCode to string
df['PostalCode'] = df['PostalCode'].astype(str)

# === Step 2: Hypothesis Testing Function ===
def safe_ttest(group_a, group_b):
    if len(group_a) < 2 or len(group_b) < 2:
        return None, 1.0
    return ttest_ind(group_a, group_b, equal_var=False)

def hypothesis_test(df, feature, group_a, group_b):
    print(f"\n--- Testing Feature: {feature} | Group A: {group_a}, Group B: {group_b} ---")

    ab_df = df[df[feature].isin([group_a, group_b])]

    # Chi-squared for Claim Frequency
    contingency = pd.crosstab(ab_df[feature], ab_df['HasClaim'])
    chi2, p1, _, _ = chi2_contingency(contingency)

    # T-test for Claim Severity (only if claim occurred)
    claims_a = ab_df[(ab_df[feature] == group_a) & (ab_df['HasClaim'])]['TotalClaims']
    claims_b = ab_df[(ab_df[feature] == group_b) & (ab_df['HasClaim'])]['TotalClaims']
    _, p2 = safe_ttest(claims_a, claims_b)

    # T-test for Margin
    margin_a = ab_df[ab_df[feature] == group_a]['Margin']
    margin_b = ab_df[ab_df[feature] == group_b]['Margin']
    _, p3 = safe_ttest(margin_a, margin_b)

    # === Print Results ===
    print(f"🟡 Claim Frequency (Chi²) p-value: {p1:.4f}")
    print("→", "Reject H₀" if p1 < 0.05 else "Fail to reject H₀")

    print(f"🟢 Claim Severity (T-test) p-value: {p2:.4f}")
    print("→", "Reject H₀" if p2 < 0.05 else "Fail to reject H₀")

    print(f"🔵 Margin Difference (T-test) p-value: {p3:.4f}")
    print("→", "Reject H₀" if p3 < 0.05 else "Fail to reject H₀")

# === Step 3: Suggest Top Categories ===
print("\n=== Top Provinces by Volume ===")
print(df['Province'].value_counts().head())

print("\n=== Top Postal Codes by Volume ===")
print(df['PostalCode'].value_counts().head())

print("\n=== Gender Counts ===")
print(df['Gender'].value_counts())

# === Step 4: Run Hypotheses ===
# 👉 Replace these values based on what's printed above if needed

# Province
hypothesis_test(df, "Province", "Gauteng", "Western Cape")

# PostalCode – replace with values printed from above
hypothesis_test(df, "PostalCode", "2000", "8000")

# Gender
hypothesis_test(df, "Gender", "Male", "Female")


KeyError: 'Gender'